# 🤖 Android Toolkit – Project Analysis & Security Review Notebook

This notebook reads **Android-Toolkit-Fixed.zip** (an Electron + React + TypeScript + ADB desktop app) and analyzes it:

1. Project structure (files, lines of code)
2. Dependencies and outdated versions
3. Map of IPC channels and ADB commands
4. Download-link inventory and risk categories
5. A test of the command validator (where it wrongly rejects input, and where it has gaps)
6. A table of review findings and summary files

> ⚠️ This is **static analysis**. The notebook does not run any command on a phone and does not open any link.

## 1️⃣ Setup and load the zip
In Google Colab you will be asked to upload the zip. In local Jupyter, keep the zip next to this notebook.

In [ ]:
import json
import re
import zipfile
from collections import Counter
from pathlib import Path
from urllib.parse import urlparse

import matplotlib.pyplot as plt
import pandas as pd

pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 140)

ZIP_NAME = "Android-Toolkit-Fixed.zip"
EXTRACT_DIR = Path("android_toolkit_src")

if not Path(ZIP_NAME).exists():
    try:
        from google.colab import files  # only exists in Google Colab
        print(f"Please upload {ZIP_NAME}")
        uploaded = files.upload()
        ZIP_NAME = next(iter(uploaded))
    except ImportError:
        raise FileNotFoundError(f"Put {ZIP_NAME} next to this notebook and run again.")

with zipfile.ZipFile(ZIP_NAME) as z:
    z.extractall(EXTRACT_DIR)

ROOT = next(p for p in EXTRACT_DIR.iterdir() if p.is_dir())
print("Project root:", ROOT)

## 2️⃣ File inventory (counts and sizes)

In [ ]:
CODE_EXT = {".ts", ".tsx", ".js", ".css", ".ejs", ".yml", ".sh", ".json", ".md"}

def area_of(rel: str) -> str:
    if rel.startswith("src/main"):
        return "main process"
    if rel.startswith("src/renderer"):
        return "renderer (UI)"
    if rel.startswith("src/__tests__"):
        return "tests"
    if rel.startswith(".erb"):
        return "build config"
    if rel.startswith(".github"):
        return "CI / GitHub"
    if rel.startswith("assets"):
        return "assets"
    return "root / other"

def count_lines(path: Path) -> int:
    try:
        return len(path.read_text(encoding="utf-8", errors="ignore").splitlines())
    except OSError:
        return 0

rows = []
for p in ROOT.rglob("*"):
    if not p.is_file():
        continue
    rel = p.relative_to(ROOT).as_posix()
    ext = p.suffix.lower() or "(none)"
    rows.append({
        "path": rel,
        "ext": ext,
        "area": area_of(rel),
        "bytes": p.stat().st_size,
        "lines": count_lines(p) if ext in CODE_EXT else 0,
    })

inv = pd.DataFrame(rows)
print(f"Total files: {len(inv)}  |  total size: {inv.bytes.sum() / 1024:.0f} KB  |  code lines: {inv.lines.sum()}")
summary_area = inv.groupby("area").agg(files=("path", "count"), lines=("lines", "sum"), kb=("bytes", lambda s: round(s.sum() / 1024)))
display(summary_area.sort_values("lines", ascending=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

by_ext = inv[inv.lines > 0].groupby("ext").lines.sum().sort_values()
by_ext.plot.barh(ax=axes[0], color="#4c78a8")
axes[0].set_title("Lines of code by file type")
axes[0].set_xlabel("lines")

by_area = inv[inv.lines > 0].groupby("area").lines.sum().sort_values()
by_area.plot.barh(ax=axes[1], color="#f58518")
axes[1].set_title("Lines of code by project area")
axes[1].set_xlabel("lines")

plt.tight_layout()
plt.show()

## 3️⃣ Dependencies
Read the dependencies from `package.json` and check which versions look outdated.

In [ ]:
pkg = json.loads((ROOT / "package.json").read_text(encoding="utf-8"))

dep_rows = []
for group in ("dependencies", "devDependencies"):
    for name, spec in pkg.get(group, {}).items():
        m = re.search(r"(\d+)", spec)
        dep_rows.append({"package": name, "spec": spec, "major": int(m.group(1)) if m else None, "group": group})

deps = pd.DataFrame(dep_rows)
print(f"App: {pkg['name']} v{pkg['version']}  |  runtime deps: {(deps.group == 'dependencies').sum()}  |  dev deps: {(deps.group == 'devDependencies').sum()}")

# Packages worth a manual upgrade check (notes written from review, versions NOT looked up online)
WATCH = {
    "electron": "Old major version, end-of-life. Upgrade is needed for security",
    "react": "React 17. Newer majors exist; plan an upgrade",
    "react-dom": "Upgrade together with react",
    "jest": "jest 27, outdated",
    "typescript": "TS 4.x, outdated",
    "electron-builder": "Check together with the Electron upgrade",
    "react-router-dom": "v5, older API",
    "octokit": "Used by the updater",
}
watch = deps[deps.package.isin(WATCH)].copy()
watch["note"] = watch.package.map(WATCH)
display(watch[["package", "spec", "note"]].reset_index(drop=True))

## 4️⃣ Map of IPC channels and ADB commands
Which routes are open from the UI (renderer) to the main process, and which ADB commands the UI sends.

In [ ]:
ipc_src = (ROOT / "src/main/api/ipcHandler.ts").read_text(encoding="utf-8")
preload_src = (ROOT / "src/main/preload.js").read_text(encoding="utf-8")

main_channels = re.findall(r"ipcMain\.on\('(\w+)'", ipc_src)
preload_send = re.findall(r"ipcRenderer\.send\('(\w+)'", preload_src)
preload_listen = re.findall(r"ipcRenderer\.on\('(\w+)'", preload_src)

print("main listens on      :", main_channels)
print("preload sends to     :", preload_send)
print("preload listens for  :", preload_listen)

# Channels the main process replies on but preload never forwards to the UI
replies = set(re.findall(r"event\.reply\('(\w+)'", ipc_src))
print("main replies on      :", sorted(replies))
print("NOT forwarded by preload:", sorted(replies - set(preload_listen)))

In [ ]:
adb_pattern = re.compile(r"adb (?:shell|connect|disconnect|devices|install|pair)[^'\"`)]*")
found = []
for f in list((ROOT / "src/renderer").rglob("*.ts*")):
    for line in f.read_text(encoding="utf-8").splitlines():
        if line.strip().startswith("//"):
            continue
        for m in adb_pattern.findall(line):
            found.append({"file": f.name, "command": m.strip().rstrip(",;")})

adb_df = pd.DataFrame(found).drop_duplicates()

def category(cmd: str) -> str:
    parts = cmd.split()
    if len(parts) > 2 and parts[1] == "shell":
        return f"shell {parts[2]}"
    return parts[1]

adb_df["category"] = adb_df.command.map(category)
print(f"{len(adb_df)} distinct ADB commands in the UI code")

counts = adb_df.category.value_counts().sort_values()
counts.plot.barh(figsize=(8, 5), color="#54a24b", title="ADB commands used by the UI (by type)")
plt.xlabel("count")
plt.tight_layout()
plt.show()

### Debloat list
How many `pm disable-user` commands does `TvActions` fire at once?

In [ ]:
debloat_src = (ROOT / "src/renderer/constants/debloatCommands.ts").read_text(encoding="utf-8")
debloat_cmds = re.findall(r"'(adb shell pm disable-user [\w.]+)'", debloat_src)
print(f"{len(debloat_cmds)} debloat commands, all fired together by one button")
print("Sample:", debloat_cmds[:3])

tv_src = (ROOT / "src/renderer/components/TvActions.tsx").read_text(encoding="utf-8")
print("Fires in parallel (.map):", "DEBLOAT_CMDS.map" in tv_src)
print("Has a confirmation dialog:", any(k in tv_src for k in ("Swal.fire", "confirm(", "showMessageBox")))

## 5️⃣ Download links
Host and risk category of each APK link. (Links are not opened; the grouping is based on the host name only.)

In [ ]:
const_src = (ROOT / "src/main/constants/index.ts").read_text(encoding="utf-8")
url_pairs = re.findall(r"^export const (\w+_URL)\s*=\s*'(https?://[^']+)'", const_src, flags=re.M)

def classify(url: str) -> str:
    host = urlparse(url).netloc
    if "github.com" in host:
        return "GitHub release (verify source)"
    if "drive.usercontent.google.com" in host or "drive.google.com" in host:
        return "Google Drive file (unverified)"
    if "mediafire" in host:
        return "MediaFire temp link (likely expired)"
    if "apkpure" in host:
        return "APK mirror site"
    if "xda-developers" in host:
        return "Forum attachment"
    return "Other third-party"

url_df = pd.DataFrame(
    [{"constant": n, "host": urlparse(u).netloc, "path_start": urlparse(u).path[:45], "risk_class": classify(u)} for n, u in url_pairs]
)
display(url_df)

url_df.risk_class.value_counts().sort_values().plot.barh(figsize=(8, 3.5), color="#e45756", title="Download links by source type")
plt.xlabel("links")
plt.tight_layout()
plt.show()

print("Hash / checksum verification present in downloadFile.ts:",
      any(k in (ROOT / "src/main/utils/downloadFile.ts").read_text() for k in ("sha256", "createHash", "checksum")))

## 6️⃣ Testing the command validator
We read the regexes from `executeCmd.ts`, make a Python copy of them, and check which commands get through.

In [ ]:
exec_src = (ROOT / "src/main/utils/executeCmd.ts").read_text(encoding="utf-8")
for name in ("SHELL_META", "ADB_SUBCOMMANDS", "IP_PORT"):
    line = next(l for l in exec_src.splitlines() if f"const {name}" in l)
    print(line.strip())

# Python port of the three regexes above (kept identical on purpose)
SHELL_META = re.compile(r"[&|;<>`$\r\n^%]")
ADB_SUBCOMMANDS = re.compile(r"^adb (connect|disconnect|devices|install|shell|kill-server|start-server)( |$)")
IP_PORT = re.compile(r"^[A-Za-z0-9.\-:\[\]]{1,255}$")

def validate_adb_command(command: str):
    if not ADB_SUBCOMMANDS.match(command):
        return "Command not allowed"
    if SHELL_META.search(command):
        return "Command contains characters that are not allowed"
    m = re.match(r"^adb connect (.+)$", command)
    if m and not IP_PORT.match(m.group(1).strip()):
        return "Invalid device address"
    return None  # None means allowed

In [ ]:
# (command, should it be allowed?, note)
cases = [
    ("adb devices", True, "normal"),
    ("adb connect 192.168.1.10:5555", True, "normal"),
    ("adb shell dumpsys battery", True, "normal"),
    ("adb connect 1.2.3.4 && calc", False, "command chaining"),
    ("adb shell ls; rm -rf /", False, "command chaining"),
    ("adb shell echo $(id)", False, "command substitution"),
    ("adb reboot", False, "not in allow-list"),
    ('adb install -r -d "C:\\apks\\R&D.apk"', True, "legit file name with &"),
    ('adb install -r -d "C:\\apks\\100%.apk"', True, "legit file name with %"),
    ('adb install -r -d "/home/u/my$app.apk"', True, "legit file name with $"),
    ("adb shell ls *", False, "* may be expanded by the host shell"),
    ("adb shell echo (x)", False, "parentheses are not blocked"),
]

rows = []
for cmd, should_allow, note in cases:
    allowed = validate_adb_command(cmd) is None
    if allowed == should_allow:
        verdict = "OK"
    elif should_allow and not allowed:
        verdict = "FALSE POSITIVE (valid input rejected)"
    else:
        verdict = "GAP (passes validator)"
    rows.append({"command": cmd, "allowed_by_validator": allowed, "note": note, "verdict": verdict})

val_df = pd.DataFrame(rows)
display(val_df)
print(val_df.verdict.value_counts().to_string())

### Safer approach: an argument array instead of a shell string
When a command is passed as a **list** instead of a string, no shell is used, so `&`, `;` and `$` stay ordinary characters. In Node.js this is `execFile(adb, [args])`. Below, Python's `subprocess` shows the same principle.

In [ ]:
import subprocess
import sys

payload = 'C:\\apks\\R&D.apk"; whoami; echo "'
result = subprocess.run(
    [sys.executable, "-c", "import sys; print(sys.argv[1:])", "install", "-r", "-d", payload],
    capture_output=True, text=True,
)
print("The whole payload arrived as ONE literal argument:")
print(result.stdout.strip())

## 7️⃣ Review findings (table)

In [ ]:
findings = [
    ("H1", "High", "constants/index.ts", "Unverified APK sources (Google Drive file, MediaFire temp link, forum/mirror links)", "Keep only verified official GitHub releases; add SHA-256 per APK"),
    ("H2", "High", "downloadFile.ts", "No status / content-type / size / hash check; borrowed User-Agent string", "Check PK header, size limit, hash; use own User-Agent"),
    ("H3", "High", "executeCmd.ts, ipcHandler.ts", "String commands via exec(); blacklist rejects valid names and misses some characters", "Use execFile with argument arrays; send structured messages from UI"),
    ("H4", "High", "package.json, main.ts", "Electron 25 end-of-life; webPreferences / openExternal / navigation not hardened", "Upgrade Electron; set contextIsolation, sandbox; allow only https in openExternal"),
    ("H5", "High", "appUpdater.ts, menu.ts, install.sh", "Updater pipes a remote script from the main branch into bash (install.sh also uses sudo)", "Use signed release assets + checksum or electron-updater"),
    ("M1", "Medium", "main.ts", "Race: adb server started before platform-tools download finishes", "await setup, then start server"),
    ("M2", "Medium", "main.ts", "tar resolved from PATH", "Use System32\\tar.exe"),
    ("M3", "Medium", "TvActions.tsx", "60 debloat commands in parallel, no confirmation, output overwritten", "Confirm dialog, sequential run, final summary"),
    ("M4", "Medium", "ipcHandler.ts", "13 copy-pasted install cases, long error messages with full command", "Table-driven installRemoteApk()"),
    ("M5", "Medium", "ipcHandler.ts", "Double click starts two downloads to the same file", "In-flight lock"),
    ("M6", "Medium", "main.ts", "kill-server on exit may disturb other ADB tools", "Kill only if this app started the server"),
    ("M7", "Medium", "constants/index.ts", "POWERSHELL_CMD quoting and cwd look fragile (not tested)", "spawn('powershell', args, {cwd})"),
    ("M8", "Medium", "package.json, __tests__", "Single render test; jest setup needs a prior build", "Unit-test validators; drop check-build-exists"),
    ("M9", "Medium", ".github/workflows", "Two workflows fire on v* tags; no lockfile / npm ci; never run", "Commit package-lock.json, merge workflows"),
    ("M10", "Medium", "package.json", "Old author/appId/placeholder repo URL", "Set own repo URL; keep MIT credit to original author"),
]
find_df = pd.DataFrame(findings, columns=["id", "severity", "where", "issue", "fix"])
display(find_df[["id", "severity", "where", "issue"]])

order = ["High", "Medium"]
find_df.severity.value_counts().reindex(order).plot.bar(color=["#e45756", "#f58518"], figsize=(4, 3), title="Findings by severity")
plt.ylabel("count")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 8️⃣ Save summary files
Here we save CSV and Markdown summaries that can be added to GitHub.

In [ ]:
OUT = Path("analysis_output")
OUT.mkdir(exist_ok=True)

find_df.to_csv(OUT / "findings.csv", index=False)
url_df.to_csv(OUT / "download_links.csv", index=False)
val_df.to_csv(OUT / "validator_tests.csv", index=False)

lines = [
    f"# Android Toolkit v{pkg['version']} - analysis summary",
    "",
    f"- Files: {len(inv)}, code lines: {int(inv.lines.sum())}",
    f"- ADB commands in UI: {len(adb_df)}, debloat commands: {len(debloat_cmds)}",
    f"- Download links: {len(url_df)} (hash check present: False)",
    f"- Validator test cases: {len(val_df)}; verdicts: {val_df.verdict.value_counts().to_dict()}",
    f"- Findings: {find_df.severity.value_counts().to_dict()}",
    "",
    "## Findings",
    "",
]
lines += [f"- **{r.id}** ({r.severity}) `{r.where}`: {r.issue}. Fix: {r.fix}" for r in find_df.itertuples()]
(OUT / "summary.md").write_text("\n".join(lines), encoding="utf-8")

print("Saved:", sorted(p.name for p in OUT.iterdir()))

## ✅ Next steps
1. Fix H1 + H2 (download links and download checks)
2. Rewrite commands with `execFile` and add unit tests
3. Fix the race condition and add a debloat confirmation
4. Fix the README, repo URL and CI
5. Finally, upgrade Electron and add a safe updater

*The notebook only reads source code. Build, lint and testing on a real device have to be done separately.*